# Dataset EDA

This notebook checks that the precomputed SHARP Doppler traces are available under `data/doppler_traces`. If the folder is missing, it downloads `doppler_traces.zip` from Google Drive and extracts it into the expected layout.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path("..").resolve()
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from download import ensure_doppler_traces

DOPPLER_DIR = ensure_doppler_traces(PROJECT_ROOT / "data")

In [ ]:
import pickle
import matplotlib.pyplot as plt

path = DOPPLER_DIR / "S1a" / "S1a_C_stream_0.txt"

with open(path, "rb") as f:
    trace = pickle.load(f)

print(trace.shape)

plt.imshow(trace.T, aspect="auto", origin="lower", cmap="viridis")
plt.xlabel("time")
plt.ylabel("Doppler bin")
plt.colorbar(label="normalized power")
plt.show()

# Training

In [ ]:
from torch.cuda import is_available
from torch.optim import Adam
from torch.nn import CrossEntropyLoss
from torch.utils.data import DataLoader
import tqdm

from models.base_model import MultiAntennaModel, SingleAntennaModel
from dataset import DopplerWindowDataset
from tqdm.auto import tqdm

EPOCHS = 25

single_antenna_model = SingleAntennaModel(num_classes=5)
model = MultiAntennaModel(single_antenna_model)

device = "cuda" if is_available() else "cpu"
model.to(device)

optimizer = Adam(
    model.parameters(),
    lr=1e-4,
    betas=(0.9, 0.999),
    eps=1e-7,
    weight_decay=0.0,
)

loss = CrossEntropyLoss()

train_dataset = DopplerWindowDataset(DOPPLER_DIR, split=(0, 0.6))
train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True)

for epoch in tqdm(range(EPOCHS), desc="Epochs"):
    model.train()
    running_loss = 0.0

    batch_bar = tqdm(train_dataloader, desc=f"Epoch {epoch + 1}/{EPOCHS}", leave=False)
    for step, (x, y) in enumerate(batch_bar, start=1):
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad()
        logits = model.forward_antennas(x)
        out = logits.reshape(-1, logits.size(-1))
        target = y.repeat_interleave(x.size(1))
        l = loss(out, target)
        l.backward()
        optimizer.step()

        running_loss += l.item()
        batch_bar.set_postfix(loss=f"{running_loss / step:.4f}")

# Eval

In [ ]:
from torch import no_grad

val_dataset = DopplerWindowDataset(DOPPLER_DIR, split=(0.6, 0.8))
val_dataloader = DataLoader(val_dataset, batch_size=32, shuffle=False)

model.eval()

val_loss = 0.0
correct = 0
total = 0

with no_grad():
    for x, y in val_dataloader:
        x, y = x.to(device), y.to(device)

        # Fused multi-antenna logits: [B, C]
        logits = model(x, fusion="sum")

        # One label per 4-antenna sample: [B]
        batch_loss = loss(logits, y)

        val_loss += batch_loss.item() * x.size(0)

        pred = logits.argmax(dim=1)
        correct += (pred == y).sum().item()
        total += y.size(0)

val_loss = val_loss / total
val_accuracy = correct / total

print(f"val_loss={val_loss:.4f} val_accuracy={val_accuracy:.4f}")
        